# Fase 0 — Análisis de Factibilidad: Material Crudo GCBA / LSA

**Pregunta estratégica:** ¿Sirve acumular estas horas de video para entrenar un modelo SignFormer?

| Sección | Pregunta |
|---------|----------|
| 1 | ¿Cuánto material hay? (clips, horas, fps, resolución) |
| 2 | ¿Es de calidad suficiente para MediaPipe? (blur, iluminación) |
| 3 | ¿Los subtítulos existen, se corresponden con los videos, y tienen riqueza de vocabulario? |
| 4 | ¿MediaPipe detecta pose + manos con confianza aceptable? |
| 5 | ¿Podemos generar un dataset mock en formato SignFormer? |
| 6 | GO / NO-GO y recomendación estratégica |

In [ ]:
import sys
sys.path.insert(0, '../scripts')

import json
import re
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
from pathlib import Path

import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import yaml

from analyze_raw import scan_folder, scan_video
from parse_subs_docx import parse_subs_docx, topic_from_text
from extract_keypoints import extract_keypoints

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

RAW_DIR     = Path(config['paths']['videos'])
SUBS_DOCX   = Path(config['paths']['subs_docx'])
DATASET_DIR = Path('../data/dataset')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print('Config cargada OK')
print(f'  Videos: {RAW_DIR}')
print(f'  Subs:   {SUBS_DOCX}')
print(f'  Existe videos: {RAW_DIR.exists()}')
print(f'  Existe subs:   {SUBS_DOCX.exists()}')

---
## Sección 1 — Inventario de videos
Escanea todos los MOV: duración, fps, resolución. Clasifica por flags del nombre de archivo.

In [ ]:
print('Escaneando videos (puede tardar ~1 min)...')
metas = scan_folder(RAW_DIR)
df = pd.DataFrame([
    {
        'filename':    m.filename,
        'video_num':   m.video_num,
        'duration_s':  m.duration_s,
        'fps':         m.fps,
        'width':       m.width,
        'height':      m.height,
        'blur_score':  m.blur_score,
        'readable':    m.readable,
        'flags':       ','.join(m.flags) if m.flags else 'ok',
        'rejected':    'rejected' in m.flags or 'redo' in m.flags,
        'title_card':  'title' in m.flags,
        'needs_review':'review' in m.flags or 'confirm' in m.flags,
    }
    for m in metas
])

print(f"Total archivos: {len(df)}")
print(f"Legibles por OpenCV: {df.readable.sum()}")
print()
print(df[['filename','duration_s','fps','width','height','flags']].to_string(index=False))

In [ ]:
usable = df[df.readable & ~df.rejected]
title_cards = usable[usable.title_card]
content     = usable[~usable.title_card]

total_s  = usable.duration_s.sum()
content_s = content.duration_s.sum()

print('=' * 55)
print('INVENTARIO DE VIDEOS')
print('=' * 55)
print(f'  Archivos totales:          {len(df):>4}')
print(f'  Legibles:                  {df.readable.sum():>4}')
print(f'  Rechazados (NO SE USA/redo): {df.rejected.sum():>4}')
print(f'  Title cards:               {df.title_card.sum():>4}')
print(f'  Para revisar:              {df.needs_review.sum():>4}')
print(f'  Clips de contenido usables:{len(content):>4}')
print()
print(f'  Duración total (todos):    {total_s/60:.1f} min  ({total_s/3600:.2f}h)')
print(f'  Duración contenido:        {content_s/60:.1f} min  ({content_s/3600:.2f}h)')
print()
print(f'  Resolución más común:      {df.width.mode()[0]}x{df.height.mode()[0]}')
print(f'  FPS más común:             {df.fps.mode()[0]:.2f}')
print(f'  Duración promedio clip:    {content.duration_s.mean():.1f}s')
print(f'  Duración mínima:           {content.duration_s.min():.1f}s')
print(f'  Duración máxima:           {content.duration_s.max():.1f}s')
print('=' * 55)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(content.duration_s, bins=20, color='steelblue', edgecolor='white')
ax1.set_xlabel('Duración (segundos)')
ax1.set_ylabel('Cantidad de clips')
ax1.set_title('Distribución de duración de clips')
ax1.axvline(content.duration_s.mean(), color='red', linestyle='--', label=f'Media: {content.duration_s.mean():.0f}s')
ax1.legend()

cats = ['Contenido\nusable', 'Title cards', 'Rechazados', 'Necesita\nrevisión']
vals = [len(content[~content.needs_review]), len(title_cards), df.rejected.sum(), df.needs_review.sum()]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
ax2.bar(cats, vals, color=colors, edgecolor='white')
ax2.set_title('Clasificación de archivos')
ax2.set_ylabel('Cantidad')
for i, v in enumerate(vals):
    ax2.text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Sección 2 — Calidad de video
**Blur score** (varianza del Laplaciano sobre frame central): en videos de señas el intérprete está en movimiento, por lo que los valores son naturalmente bajos (15-30). Lo que importa es detectar **outliers** — clips con blur << al resto (cámara desenfocada, frame negro, takes fallidas).

In [ ]:
BLUR_THRESHOLD = 10  # outliers por debajo de este valor son problemáticos

low_blur = content[content.blur_score < BLUR_THRESHOLD]
high_blur = content[content.blur_score >= BLUR_THRESHOLD]

print(f'NOTA: blur scores de 15-30 son NORMALES en videos de señas (movimiento de manos).')
print(f'Se detectan outliers con blur < {BLUR_THRESHOLD} (cámara desenfocada / takes fallidas).')
print()
print(f'Clips OK (blur >= {BLUR_THRESHOLD}): {len(high_blur)} / {len(content)}')
print(f'Clips problemáticos (blur < {BLUR_THRESHOLD}): {len(low_blur)}')
if len(low_blur) > 0:
    print()
    print('Clips con posible problema de calidad:')
    print(low_blur[['filename','blur_score','duration_s']].to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(content.blur_score, bins=20, color='steelblue', edgecolor='white')
ax1.axvline(BLUR_THRESHOLD, color='red', linestyle='--', label=f'Outlier threshold {BLUR_THRESHOLD}')
ax1.set_xlabel('Blur score (varianza Laplaciano)')
ax1.set_ylabel('Clips')
ax1.set_title('Distribución de nitidez\n(rango 15-30 es normal para videos de señas)')
ax1.legend()

# Mostrar frame de muestra del clip más borroso y el más nítido
def grab_mid_frame(filename):
    path = RAW_DIR / filename
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None

worst = content.nsmallest(1, 'blur_score').iloc[0]
best  = content.nlargest(1, 'blur_score').iloc[0]

worst_frame = grab_mid_frame(worst.filename)
best_frame  = grab_mid_frame(best.filename)

if worst_frame is not None and best_frame is not None:
    ax2.axis('off')
    fig2, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].imshow(worst_frame)
    axes[0].set_title(f'Blur más bajo (outlier)\n{worst.filename[:40]}\nblur={worst.blur_score:.1f}', fontsize=9)
    axes[0].axis('off')
    axes[1].imshow(best_frame)
    axes[1].set_title(f'Blur más alto (referencia)\n{best.filename[:40]}\nblur={best.blur_score:.1f}', fontsize=9)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

plt.tight_layout()
plt.show()

---
## Sección 3 — Análisis de subtítulos
Parsea el DOCX, matchea con videos por N°, analiza cobertura y riqueza léxica.

In [ ]:
subs = parse_subs_docx(SUBS_DOCX)
df_subs = pd.DataFrame([
    {
        'video_num': s.video_num,
        'note':      s.note,
        'text':      s.text,
        'topic':     topic_from_text(s.text),
        'word_count': len(s.text.split()),
    }
    for s in subs
])

print(f'Subtítulos parseados: {len(df_subs)}')
print()
print(df_subs[['video_num','topic','word_count','text']].assign(
    text=df_subs.text.str[:70]
).to_string(index=False))

In [ ]:
# Matching subtítulos ↔ videos
video_nums_with_subs = set(df_subs.video_num)
content_nums = set(content.video_num.dropna().astype(int))

matched   = content_nums & video_nums_with_subs
no_sub    = content_nums - video_nums_with_subs
sub_no_vid = video_nums_with_subs - content_nums

print('COBERTURA SUBTÍTULOS ↔ VIDEOS')
print('=' * 45)
print(f'  Videos de contenido:          {len(content_nums)}')
print(f'  Subtítulos en DOCX:           {len(video_nums_with_subs)}')
print(f'  Videos CON subtítulo:         {len(matched)}')
print(f'  Videos SIN subtítulo:         {len(no_sub)}')
print(f'  Subs sin video correspondiente:{len(sub_no_vid)}')
print()
if no_sub:
    no_sub_names = content[content.video_num.isin(no_sub)][['filename','duration_s']]
    print(f'Videos sin subtítulo ({len(no_sub)}):')
    print(no_sub_names.to_string(index=False))

if sub_no_vid:
    print(f'\nSubs sin video: N° {sorted(sub_no_vid)}')

# Tiempo cubierto con subtítulo
content_matched = content[content.video_num.isin(matched)]
print(f'\nDuración contenido con sub:  {content_matched.duration_s.sum()/60:.1f} min')
print(f'Duración contenido sin sub:  {content[content.video_num.isin(no_sub)].duration_s.sum()/60:.1f} min')

In [ ]:
# Riqueza léxica
import unicodedata

STOP = {'de','la','el','en','y','a','que','es','los','las','se','con','para',
        'un','una','por','del','al','su','sus','le','si','o','no','lo','más',
        'son','sus','como','una','este','esta','ese','esa','puede','pueden',
        'tienen','tiene','hay','ser','podes','te','tu','tus','mi','me','nos'}

def normalize(w):
    w = w.lower()
    w = ''.join(c for c in unicodedata.normalize('NFD', w) if unicodedata.category(c) != 'Mn')
    return re.sub(r'[^a-z]', '', w)

all_text = ' '.join(df_subs.text)
raw_words = all_text.split()
words = [normalize(w) for w in raw_words if normalize(w) and normalize(w) not in STOP]

total_words = len(raw_words)
vocab = set(words)
freq = Counter(words).most_common(30)

print('RIQUEZA LÉXICA')
print('=' * 45)
print(f'  Palabras totales:     {total_words}')
print(f'  Vocabulario único:    {len(vocab)}')
print(f'  TTR (type/token):     {len(vocab)/len(words):.3f}  (>0.3 = rico)')
print()
print('  Top 30 palabras clave:')
print('  ' + ', '.join(f'{w}({c})' for w, c in freq))

# Temas
print()
topic_counts = df_subs.topic.value_counts()
print('DISTRIBUCIÓN DE TEMAS:')
print(topic_counts.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Word frequency
top_words, top_counts = zip(*freq[:20])
axes[0].barh(list(reversed(top_words)), list(reversed(top_counts)), color='steelblue')
axes[0].set_title('Top 20 palabras (sin stopwords)')
axes[0].set_xlabel('Frecuencia')

# Topics pie
axes[1].pie(topic_counts.values, labels=topic_counts.index, autopct='%1.0f%%',
            colors=plt.cm.Set3.colors[:len(topic_counts)])
axes[1].set_title('Distribución de temas')

plt.tight_layout()
plt.show()

---
## Sección 4 — Probe de MediaPipe
Corre MediaPipe Holistic sobre 5 clips representativos. Reporta: ¿detecta pose? ¿manos? ¿con qué confianza?

In [ ]:
# Seleccionar 5 clips del contenido con subtítulo (los más representativos por duración)
probe_candidates = content_matched.sort_values('duration_s').iloc[
    [0, len(content_matched)//4, len(content_matched)//2, 3*len(content_matched)//4, -1]
]

mp_holistic = mp.solutions.holistic
mp_draw     = mp.solutions.drawing_utils

probe_results = []

for _, row in probe_candidates.iterrows():
    path = RAW_DIR / row.filename
    cap  = cv2.VideoCapture(str(path))
    fps  = cap.get(cv2.CAP_PROP_FPS) or 30
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # sample 10 frames spread across the clip
    sample_frames = np.linspace(0, total-1, 10, dtype=int)
    frame_results = []

    with mp_holistic.Holistic(
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ) as holistic:
        for fi in sample_frames:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
            ret, frame = cap.read()
            if not ret:
                continue
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = holistic.process(rgb)
            pose_conf = np.mean([lm.visibility for lm in res.pose_landmarks.landmark]) if res.pose_landmarks else 0.0
            frame_results.append({
                'frame': int(fi),
                'has_pose':       res.pose_landmarks is not None,
                'has_left_hand':  res.left_hand_landmarks is not None,
                'has_right_hand': res.right_hand_landmarks is not None,
                'pose_conf':      pose_conf,
                'frame_rgb':      rgb.copy(),
                'mp_results':     res,
            })

    cap.release()

    probe_results.append({
        'filename':   row.filename,
        'duration_s': row.duration_s,
        'frames':     frame_results,
        'pose_pct':   np.mean([f['has_pose'] for f in frame_results]) * 100,
        'lhand_pct':  np.mean([f['has_left_hand'] for f in frame_results]) * 100,
        'rhand_pct':  np.mean([f['has_right_hand'] for f in frame_results]) * 100,
        'conf_avg':   np.mean([f['pose_conf'] for f in frame_results if f['has_pose']] or [0]),
    })
    print(f"  {row.filename[:45]:45s}  pose={probe_results[-1]['pose_pct']:5.0f}%  lhand={probe_results[-1]['lhand_pct']:5.0f}%  rhand={probe_results[-1]['rhand_pct']:5.0f}%  conf={probe_results[-1]['conf_avg']:.3f}")

print('\nDone.')

In [ ]:
# Tabla resumen de MediaPipe probe
df_probe = pd.DataFrame([
    {
        'clip':       r['filename'][:40],
        'dur(s)':     r['duration_s'],
        'pose %':     f"{r['pose_pct']:.0f}%",
        'l_hand %':   f"{r['lhand_pct']:.0f}%",
        'r_hand %':   f"{r['rhand_pct']:.0f}%",
        'conf_avg':   f"{r['conf_avg']:.3f}",
    }
    for r in probe_results
])
print(df_probe.to_string(index=False))

# Visualizar overlay en el frame más representativo de cada clip
fig, axes = plt.subplots(1, len(probe_results), figsize=(4*len(probe_results), 5))
if len(probe_results) == 1:
    axes = [axes]

for ax, pr in zip(axes, probe_results):
    mid = pr['frames'][len(pr['frames'])//2]
    annotated = mid['frame_rgb'].copy()
    res = mid['mp_results']
    mp_draw.draw_landmarks(annotated, res.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
    mp_draw.draw_landmarks(annotated, res.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp_draw.draw_landmarks(annotated, res.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    ax.imshow(annotated)
    ax.set_title(f"{pr['filename'][:25]}\nconf={pr['conf_avg']:.2f}", fontsize=8)
    ax.axis('off')

plt.suptitle('MediaPipe Holistic — overlay de keypoints (muestra)', fontsize=11)
plt.tight_layout()
plt.show()

---
## Sección 5 — Dataset mock para SignFormer

**¿Qué espera SignFormer?**

SignFormer (y modelos similares: SMKD, CorrNet, SignBERT) toman como entrada secuencias de keypoints:
- Forma: `(T, N_joints, 3)` donde T = frames, N_joints = 75 (pose: 33 + mano izq: 21 + mano der: 21)
- Label: glosa o sentencia en texto

La diferencia crítica: algunos modelos necesitan **glosas** (signos individuales etiquetados), otros aceptan **sentencias** (el subtítulo completo como label).

Este mock genera una entrada por clip (label = subtítulo completo, nivel sentencia).

In [ ]:
N_MOCK = 3  # generar N clips en el dataset mock

sub_lookup = {s.video_num: s for s in parse_subs_docx(SUBS_DOCX)}

mock_clips = content_matched.sort_values('duration_s').head(N_MOCK)
dataset = []

for _, row in mock_clips.iterrows():
    path = RAW_DIR / row.filename
    sub  = sub_lookup.get(int(row.video_num))

    cap   = cv2.VideoCapture(str(path))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    print(f'  Extrayendo keypoints: {row.filename[:45]} ...')
    kp = extract_keypoints(path, config, frame_start=0, frame_end=total)

    # Armar tensor (T, 75, 3) para SignFormer
    # 33 pose + 21 left_hand + 21 right_hand = 75 joints
    def kp_to_row(lm_list, n_joints):
        if lm_list is None:
            return [[0.0, 0.0, 0.0]] * n_joints
        return [[lm['x'], lm['y'], lm.get('z', 0.0)] for lm in lm_list[:n_joints]]

    frames_tensor = []
    for f in kp['frames']:
        row_joints = (
            kp_to_row(f['pose'],       33) +
            kp_to_row(f['left_hand'],  21) +
            kp_to_row(f['right_hand'], 21)
        )
        frames_tensor.append(row_joints)

    entry = {
        'id':     f"gcba_{row.video_num:04d}",
        'label':  sub.text if sub else '',
        'source': row.filename,
        'metadata': {
            'fps':         fps,
            'duration_s':  row.duration_s,
            'resolution':  f"{int(row.width)}x{int(row.height)}",
            'n_frames':    len(frames_tensor),
            'n_joints':    75,
            'label_type':  'sentence',  # no glosa — nivel sentencia
            'language':    'LSA',
            'topic':       topic_from_text(sub.text) if sub else '',
            'confidence_avg': kp['confidence_avg'],
        },
        # shape: (T, 75, 3) — (frames, joints, [x,y,z])
        'keypoints': frames_tensor,
    }
    dataset.append(entry)
    print(f'    → T={len(frames_tensor)} frames, conf={kp["confidence_avg"]:.3f}, label: "{sub.text[:60]}"')

# Guardar
out = DATASET_DIR / 'mock_signformer.json'
with open(out, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print(f'\nDataset mock guardado: {out}')
print(f'Entradas: {len(dataset)}')
print(f'Estructura por entrada: id, label, source, metadata, keypoints(T×75×3)')

In [ ]:
# Visualizar secuencia de confianza del primer clip del mock
entry = dataset[0]
T = entry['metadata']['n_frames']
fps = entry['metadata']['fps']

# Extraer visibilidad de pose (joint 0 = nariz) como proxy de confianza
# Los joints de pose tienen z como 'depth' — usamos magnitud como proxy
pose_present = [
    1 if any(j[0] != 0 or j[1] != 0 for j in frame_joints[:33]) else 0
    for frame_joints in entry['keypoints']
]
times = [i / fps for i in range(T)]

fig, axes = plt.subplots(2, 1, figsize=(13, 5))

axes[0].fill_between(times, pose_present, alpha=0.6, color='steelblue')
axes[0].set_ylabel('Pose detectada')
axes[0].set_title(f'Clip: {entry["id"]} — "{entry["label"][:60]}"')
axes[0].set_ylim(-0.1, 1.3)

rhand = [
    1 if any(j[0] != 0 for j in frame_joints[54:75]) else 0
    for frame_joints in entry['keypoints']
]
lhand = [
    1 if any(j[0] != 0 for j in frame_joints[33:54]) else 0
    for frame_joints in entry['keypoints']
]
axes[1].fill_between(times, rhand, alpha=0.6, color='#e74c3c', label='Mano derecha')
axes[1].fill_between(times, lhand, alpha=0.6, color='#2ecc71', label='Mano izquierda')
axes[1].set_ylabel('Mano detectada')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylim(-0.1, 1.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Estructura del dataset mock:')
preview = {k: v for k, v in entry.items() if k != 'keypoints'}
preview['keypoints'] = f'array ({T}, 75, 3)'
print(json.dumps(preview, indent=2, ensure_ascii=False))

---
## Sección 6 — GO / NO-GO y recomendación estratégica

In [ ]:
# Métricas agregadas de MediaPipe
avg_pose_pct  = np.mean([r['pose_pct'] for r in probe_results])
avg_lhand_pct = np.mean([r['lhand_pct'] for r in probe_results])
avg_rhand_pct = np.mean([r['rhand_pct'] for r in probe_results])
avg_conf      = np.mean([r['conf_avg'] for r in probe_results])

quality_ok    = content[content.blur_score >= BLUR_THRESHOLD].shape[0] / len(content) > 0.7
sub_coverage  = len(matched) / len(content_nums)
ttr           = len(vocab) / len(words)
n_topics      = df_subs.topic.nunique()

checks = [
    ('Clips de contenido usable', len(content), '>= 30', len(content) >= 30),
    ('Duración total contenido (min)', f'{content_s/60:.1f}', '>= 15', content_s >= 900),
    ('Resolución 1080p', f'{df.width.mode()[0]}x{df.height.mode()[0]}', '1920x1080', df.width.mode()[0] >= 1280),
    ('FPS suficiente', f'{df.fps.mode()[0]:.0f}', '>= 25', df.fps.mode()[0] >= 25),
    ('Calidad video (blur OK)', f'{quality_ok}', '> 70% clips OK', quality_ok),
    ('Cobertura subtítulos', f'{sub_coverage:.0%}', '>= 50%', sub_coverage >= 0.5),
    ('Riqueza léxica (TTR)', f'{ttr:.3f}', '>= 0.3', ttr >= 0.3),
    ('Diversidad temática', f'{n_topics} temas', '>= 5', n_topics >= 5),
    ('Pose MediaPipe', f'{avg_pose_pct:.0f}%', '>= 80%', avg_pose_pct >= 80),
    ('Manos MediaPipe', f'L:{avg_lhand_pct:.0f}% R:{avg_rhand_pct:.0f}%', '>= 50%', max(avg_lhand_pct, avg_rhand_pct) >= 50),
    ('Confianza MediaPipe (pose)', f'{avg_conf:.3f}', '>= 0.6', avg_conf >= 0.6),
]

print('=' * 65)
print('FACTIBILIDAD — RESUMEN GO/NO-GO')
print('=' * 65)
for label, value, threshold, ok in checks:
    status = '✓' if ok else '✗'
    print(f'  {status}  {label:<35} {str(value):>10}  (umbral: {threshold})')

passed = sum(ok for _, _, _, ok in checks)
total  = len(checks)
print()
print(f'  Criterios cumplidos: {passed}/{total}')
print()

if passed >= total * 0.75:
    verdict = 'GO  — Material usable para pipeline de entrenamiento'
    color   = '\033[92m'  # green
else:
    verdict = 'CONDICIONAL — Revisar criterios fallidos antes de avanzar'
    color   = '\033[93m'  # yellow

print(f'  DECISIÓN: {verdict}')
print('=' * 65)
print()
print('RECOMENDACIÓN ESTRATÉGICA:')
print()
print('  1. CORTO PLAZO: Este material sirve como PRUEBA DE CONCEPTO.')
print('     Con ~30-50 clips etiquetados a nivel sentencia se puede:')
print('     • Fine-tunear un modelo pre-entrenado (transfer learning)')
print('     • Establecer métricas de baseline')
print('     • Validar que el pipeline funciona end-to-end')
print()
print('  2. PARA ENTRENAR SignFormer desde cero se necesita:')
print('     • >>1000 horas de video (PHOENIX-2014: ~80h, CSL: ~100h)')
print('     • Anotaciones a nivel GLOSA (no solo sentencia)')
print('     • Este material: útil como seed, no suficiente como dataset principal')
print()
print('  3. SOBRE "ACUMULAR HORAS DE VIDEO":')
print('     → SÍ vale la pena si:')
print('       - El formato es consistente (mismo intérprete, mismo fondo)')
print('       - Los subtítulos pueden sincronizarse')
print('       - Pueden conseguirse al menos 5-10h de material similar')
print('     → NO es suficiente si:')
print('       - Solo hay horas de video sin anotación de glosas')
print('       - No hay forma de alinear sub↔video con precisión de frame')
print()
print('  4. PRÓXIMO PASO CRÍTICO: Sync sub↔video')
print('     El cuello de botella no es la cantidad de video sino la calidad')
print('     de la alineación temporal. Sin sync precisa, los keypoints no')
print('     corresponden a las glosas/palabras → el modelo no aprende.')